# H2 Hypothesis Testing: Pandemic Effect on ED Length of Stay

This notebook evaluates whether reported median emergency department length of stay differed between the pandemic-affected fiscal year **2020–2021** and the preceding fiscal years.

- **Dataset:** `data/Explorer Dataset/ED_Visits.csv`
- **Hypothesis test:** two-sided Mann-Whitney U test
- **Null hypothesis (H₀):** There is no significant difference in reported median ED length of stay between fiscal year 2020–2021 and preceding fiscal years.
- **Alternative hypothesis (H₁):** A significant difference exists.
- **Significance level:** α = 0.05

The notebook follows the same overall workflow as the H1 testing notebook: data loading, dataset inspection, validation, descriptive summaries, hypothesis testing, effect size, and a consistent set of visualizations.

In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt

from pathlib import Path

from scipy import stats

from IPython.display import display

# Display settings
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 140)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

print("Libraries imported successfully.")

In [ ]:
# Locate the repository root and Explorer Dataset folder
possible_roots = [Path.cwd()] + list(Path.cwd().parents[:6])

repo_root = None

for candidate in possible_roots:

    if (candidate / "data" / "Explorer Dataset").exists():

        repo_root = candidate
        break

# Fallback to the project location used in the capstone environment
if repo_root is None:

    repo_root = Path(
        r"d:\term 5 capstone\capstone offical github repo\Capstone_Project-DAMO-6994-"
    )

INPUT_FOLDER = repo_root / "data" / "Explorer Dataset"

FILE_PATH = INPUT_FOLDER / "ED_Visits.csv"

print("Resolved root:")
print(repo_root)

print("\nInput folder:")
print(INPUT_FOLDER)

print("\nInput file:")
print(FILE_PATH)

print("\nFile exists:", FILE_PATH.exists())

In [ ]:
assert FILE_PATH.exists(), (
    "ED_Visits.csv not found. "
    "Update FILE_PATH before proceeding."
)

df = pd.read_csv(FILE_PATH)

print("Dataset loaded successfully.")

In [ ]:
print("FIRST 10 ROWS")
print("-" * 80)

display(df.head(10))

In [ ]:
print("LAST 10 ROWS")
print("-" * 80)

display(df.tail(10))

In [ ]:
print("DATASET SHAPE")
print("-" * 80)

print(f"Rows    : {df.shape[0]:,}")
print(f"Columns : {df.shape[1]:,}")

In [ ]:
print("COLUMN NAMES")
print("-" * 80)

for i, column in enumerate(df.columns, start=1):

    print(f"{i:02d}. {column}")

In [ ]:
print("DATA TYPES")
print("-" * 80)

display(
    pd.DataFrame({
        "Column": df.columns,
        "Data_Type": df.dtypes.astype(str).values
    })
)

In [ ]:
print("MISSING VALUES")
print("-" * 80)

missing_summary = pd.DataFrame({

    "Column": df.columns,

    "Missing_Count": [
        df[col].isna().sum()
        for col in df.columns
    ],

    "Missing_Percentage": [
        df[col].isna().mean() * 100
        for col in df.columns
    ]

})

missing_summary["Missing_Percentage"] = (
    missing_summary["Missing_Percentage"]
    .round(2)
)

display(missing_summary)

In [ ]:
duplicate_count = df.duplicated().sum()

print("DUPLICATE RECORDS")
print("-" * 80)

print(f"Duplicate rows: {duplicate_count:,}")

print(
    f"Duplicate percentage: "
    f"{duplicate_count / len(df) * 100:.2f}%"
)

In [ ]:
print("NUMERICAL SUMMARY")
print("-" * 80)

display(
    df.describe(
        include="all"
    ).T
)

In [ ]:
print("FISCAL YEAR COVERAGE")
print("-" * 80)

fiscal_years = (
    df[
        ["fiscal_year_start", "fiscal_year"]
    ]
    .drop_duplicates()
    .sort_values("fiscal_year_start")
)

display(fiscal_years)

print("\nRecords per fiscal year")

display(
    df["fiscal_year"]
    .value_counts(dropna=False)
    .sort_index()
    .rename_axis("fiscal_year")
    .reset_index(name="record_count")
)

In [ ]:
print("FISCAL YEAR VS VISIT DISPOSITION")
print("-" * 80)

fiscal_disposition_check = pd.crosstab(
    df["fiscal_year_start"],
    df["visit_disposition"]
)

display(fiscal_disposition_check)

In [ ]:
print("ED VISITS AND LOS SUMMARY")
print("-" * 80)

selected_columns = [
    "fiscal_year",
    "fiscal_year_start",
    "visit_disposition",
    "admission_status",
    "ed_visits",
    "median_los_minutes",
    "median_los_hours"
]

display(
    df[selected_columns]
    .describe(include="all")
    .T
)

In [ ]:
print("INVALID VALUE CHECKS")
print("=" * 80)

print(
    "ED visits <= 0:",
    (df["ed_visits"] <= 0).sum()
)

print(
    "LOS < 0:",
    (df["median_los_minutes"] < 0).sum()
)

print(
    "Missing fiscal year:",
    df["fiscal_year"].isna().sum()
)

print(
    "Missing fiscal year start:",
    df["fiscal_year_start"].isna().sum()
)

print(
    "Missing ED visits:",
    df["ed_visits"].isna().sum()
)

print(
    "Missing median LOS:",
    df["median_los_minutes"].isna().sum()
)

In [ ]:
def weighted_mean(values, weights):

    values = np.asarray(values, dtype=float)
    weights = np.asarray(weights, dtype=float)

    valid = (
        np.isfinite(values) &
        np.isfinite(weights) &
        (weights > 0)
    )

    values = values[valid]
    weights = weights[valid]

    if len(values) == 0:
        return np.nan

    return np.average(
        values,
        weights=weights
    )


def weighted_median(values, weights):

    values = np.asarray(values, dtype=float)
    weights = np.asarray(weights, dtype=float)

    valid = (
        np.isfinite(values) &
        np.isfinite(weights) &
        (weights > 0)
    )

    values = values[valid]
    weights = weights[valid]

    if len(values) == 0:
        return np.nan

    order = np.argsort(values)

    values = values[order]
    weights = weights[order]

    cumulative_weights = np.cumsum(weights)

    midpoint = weights.sum() / 2

    return values[
        np.searchsorted(
            cumulative_weights,
            midpoint
        )
    ]

In [ ]:
print("WEIGHTED DESCRIPTIVE SUMMARY")
print("-" * 80)

weighted_summary = (
    df
    .dropna(
        subset=[
            "fiscal_year",
            "ed_visits",
            "median_los_minutes"
        ]
    )
    .query("ed_visits > 0 and median_los_minutes >= 0")
    .groupby("fiscal_year")
    .apply(
        lambda x: pd.Series({

            "records":
                len(x),

            "total_ed_visits":
                x["ed_visits"].sum(),

            "weighted_mean_los_minutes":
                weighted_mean(
                    x["median_los_minutes"],
                    x["ed_visits"]
                ),

            "weighted_median_los_minutes":
                weighted_median(
                    x["median_los_minutes"],
                    x["ed_visits"]
                ),

            "minimum_los_minutes":
                x["median_los_minutes"].min(),

            "maximum_los_minutes":
                x["median_los_minutes"].max()

        }),
        include_groups=False
    )
    .reset_index()
)

display(weighted_summary)

In [ ]:
required_columns = [
    "fiscal_year",
    "fiscal_year_start",
    "ed_visits",
    "median_los_minutes"
]

print("H2 DATA VALIDATION")
print("=" * 80)

for column in required_columns:

    print(
        f"{column:25} : "
        f"{'FOUND' if column in df.columns else 'MISSING'}"
    )


analysis_df = df[
    required_columns
].copy()

analysis_df = analysis_df.dropna(
    subset=required_columns
)

analysis_df = analysis_df[
    (analysis_df["ed_visits"] > 0) &
    (analysis_df["median_los_minutes"] >= 0)
].copy()


print("\nAfter validation:")

print(
    f"Records available : {len(analysis_df):,}"
)

print(
    f"Fiscal year groups: "
    f"{analysis_df['fiscal_year'].nunique()}"
)

print(
    f"Total ED visits   : "
    f"{analysis_df['ed_visits'].sum():,.0f}"
)

print(
    f"Missing values    : "
    f"{analysis_df.isna().sum().sum()}"
)

In [ ]:
# ============================================================
# H2 ANALYSIS DATASET
# ============================================================

PANDEMIC_YEAR = "2020-2021"
PANDEMIC_START = 2020

h2_df = analysis_df[
    analysis_df["fiscal_year_start"] <= PANDEMIC_START
].copy()

# Keep only the variables required for H2
h2_df = h2_df[
    [
        "fiscal_year",
        "fiscal_year_start",
        "ed_visits",
        "median_los_minutes"
    ]
].copy()

# Define the pandemic and preceding fiscal-year groups
pandemic_df = h2_df[
    h2_df["fiscal_year"] == PANDEMIC_YEAR
].copy()

pre_pandemic_df = h2_df[
    h2_df["fiscal_year_start"] < PANDEMIC_START
].copy()

print("=" * 80)
print("H2 ANALYSIS DATASET")
print("=" * 80)

print(f"Original records       : {len(df):,}")
print(f"Validated records      : {len(analysis_df):,}")
print(f"Records used for H2    : {len(h2_df):,}")

print(f"\nPandemic records       : {len(pandemic_df):,}")
print(f"Pre-pandemic records   : {len(pre_pandemic_df):,}")

print("\nPandemic fiscal year:")
print(sorted(pandemic_df["fiscal_year"].unique()))

print("\nPre-pandemic fiscal years:")
print(sorted(pre_pandemic_df["fiscal_year"].unique()))

print("\nExcluded fiscal years from H2:")
excluded_years = sorted(
    set(analysis_df["fiscal_year"].unique())
    - set(h2_df["fiscal_year"].unique())
)

for year in excluded_years:
    print(f"  {year}")

In [ ]:
print("=" * 80)
print("WEIGHT VALIDATION")
print("=" * 80)

print("ED Visits data type:")
print(h2_df["ed_visits"].dtype)

print("\nNon-integer ED visit values:")

non_integer_weights = (
    h2_df["ed_visits"] % 1 != 0
).sum()

print(non_integer_weights)

print("\nMinimum ED visits:")
print(h2_df["ed_visits"].min())

print("\nMaximum ED visits:")
print(h2_df["ed_visits"].max())

print("\nTotal ED visits represented:")
print(
    f"{h2_df['ed_visits'].sum():,.0f}"
)

In [ ]:
print("H2 GROUP SUMMARY")
print("-" * 80)

def group_summary(df_group):

    return {

        "records":
            len(df_group),

        "total_ed_visits":
            df_group["ed_visits"].sum(),

        "mean_los_minutes":
            df_group["median_los_minutes"].mean(),

        "median_los_minutes":
            df_group["median_los_minutes"].median(),

        "std_los_minutes":
            df_group["median_los_minutes"].std(),

        "weighted_mean_los_minutes":
            weighted_mean(
                df_group["median_los_minutes"],
                df_group["ed_visits"]
            ),

        "weighted_median_los_minutes":
            weighted_median(
                df_group["median_los_minutes"],
                df_group["ed_visits"]
            ),

        "minimum_los_minutes":
            df_group["median_los_minutes"].min(),

        "maximum_los_minutes":
            df_group["median_los_minutes"].max()

    }


h2_summary = pd.DataFrame([

    {
        "group": "Pandemic (2020-2021)",
        **group_summary(pandemic_df)
    },

    {
        "group": "Pre-Pandemic (preceding fiscal years)",
        **group_summary(pre_pandemic_df)
    }

])

display(h2_summary)

In [ ]:
print("FISCAL YEAR MEDIAN LOS BREAKDOWN")
print("-" * 80)

fiscal_year_summary = (
    h2_df
    .groupby("fiscal_year")
    .agg(
        records=("median_los_minutes", "count"),
        total_ed_visits=("ed_visits", "sum"),
        mean_los_minutes=("median_los_minutes", "mean"),
        median_los_minutes=("median_los_minutes", "median"),
        std_los_minutes=("median_los_minutes", "std"),
        minimum_los_minutes=("median_los_minutes", "min"),
        maximum_los_minutes=("median_los_minutes", "max")
    )
    .reset_index()
)

display(
    fiscal_year_summary
    .sort_values("fiscal_year")
)

In [ ]:
# ============================================================
# MANN-WHITNEY U HYPOTHESIS TEST
# ============================================================

print("=" * 80)
print("H2 HYPOTHESIS TEST: MANN-WHITNEY U")
print("=" * 80)

pandemic_values = (
    pandemic_df["median_los_minutes"]
    .dropna()
    .values
)

pre_pandemic_values = (
    pre_pandemic_df["median_los_minutes"]
    .dropna()
    .values
)

assert len(pandemic_values) >= 2, (
    "Insufficient pandemic records for hypothesis testing."
)

assert len(pre_pandemic_values) >= 2, (
    "Insufficient pre-pandemic records for hypothesis testing."
)

u_stat, p_value = stats.mannwhitneyu(
    pandemic_values,
    pre_pandemic_values,
    alternative="two-sided"
)

ALPHA = 0.05

print(
    f"U statistic: {u_stat:.6f}"
)

print(
    f"P-value: {p_value:.10f}"
)

print(
    f"Pandemic sample size: "
    f"{len(pandemic_values):,}"
)

print(
    f"Pre-pandemic sample size: "
    f"{len(pre_pandemic_values):,}"
)

print(
    f"Significance level (α): "
    f"{ALPHA}"
)

if p_value < ALPHA:

    print("\nDecision: REJECT H₀")

    print(
        "\nConclusion:"
    )

    print(
        "There is statistically significant evidence "
        "that reported median ED length of stay differs "
        "between fiscal year 2020-2021 and the preceding "
        "fiscal years."
    )

else:

    print("\nDecision: FAIL TO REJECT H₀")

    print(
        "\nConclusion:"
    )

    print(
        "There is insufficient statistical evidence "
        "to conclude that reported median ED length "
        "of stay differs between fiscal year 2020-2021 "
        "and the preceding fiscal years."
    )

In [ ]:
# ============================================================
# H2 EFFECT SIZE
# ============================================================

# Rank-biserial correlation is a common effect-size measure
# for a Mann-Whitney U comparison.

n1 = len(pandemic_values)
n2 = len(pre_pandemic_values)

rank_biserial = (
    (2 * u_stat) /
    (n1 * n2)
) - 1

print("H2 EFFECT SIZE")
print("-" * 80)

print(
    f"Rank-biserial correlation: "
    f"{rank_biserial:.6f}"
)

In [ ]:
h2_result_table = pd.DataFrame({

    "Hypothesis": [
        "H2"
    ],

    "Research_Question": [
        "Does reported median ED LOS differ between fiscal year 2020-2021 and preceding fiscal years?"
    ],

    "Test": [
        "Mann-Whitney U (two-sided)"
    ],

    "U_Statistic": [
        u_stat
    ],

    "Pandemic_Sample_Size": [
        n1
    ],

    "Pre_Pandemic_Sample_Size": [
        n2
    ],

    "P_Value": [
        p_value
    ],

    "Rank_Biserial_Correlation": [
        rank_biserial
    ],

    "Alpha": [
        ALPHA
    ],

    "Decision": [
        "Reject H0"
        if p_value < ALPHA
        else "Fail to Reject H0"
    ]

})

display(h2_result_table)

In [ ]:
# Visualization settings and ordered fiscal-year data

period_labels = [
    "Pre-Pandemic",
    "Pandemic 2020-2021"
]

period_dfs = [
    pre_pandemic_df,
    pandemic_df
]

period_summary = pd.DataFrame({

    "period": period_labels,

    "records": [
        len(pre_pandemic_df),
        len(pandemic_df)
    ],

    "total_ed_visits": [
        pre_pandemic_df["ed_visits"].sum(),
        pandemic_df["ed_visits"].sum()
    ],

    "weighted_mean_los": [
        weighted_mean(
            pre_pandemic_df["median_los_minutes"],
            pre_pandemic_df["ed_visits"]
        ),
        weighted_mean(
            pandemic_df["median_los_minutes"],
            pandemic_df["ed_visits"]
        )
    ],

    "weighted_median_los": [
        weighted_median(
            pre_pandemic_df["median_los_minutes"],
            pre_pandemic_df["ed_visits"]
        ),
        weighted_median(
            pandemic_df["median_los_minutes"],
            pandemic_df["ed_visits"]
        )
    ]

})

print("H2 visualization data prepared.")

display(period_summary)

In [ ]:
# NUMBER OF AGGREGATE RECORDS BY PERIOD

record_counts = period_summary["records"].values

plt.figure(figsize=(10, 6))

bars = plt.bar(
    period_labels,
    record_counts
)

plt.title(
    "Aggregate Records: Pre-Pandemic vs Pandemic"
)

plt.xlabel(
    "Period"
)

plt.ylabel(
    "Number of Aggregate Records"
)

for bar, value in zip(
    bars,
    record_counts
):

    plt.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height(),
        f"{value:,}",
        ha="center",
        va="bottom"
    )

plt.tight_layout()

plt.show()

In [ ]:
# TOTAL ED VISITS BY PERIOD

visit_summary = period_summary["total_ed_visits"].values

plt.figure(figsize=(10, 6))

bars = plt.bar(
    period_labels,
    visit_summary
)

plt.title(
    "Total Reported ED Visits by Period"
)

plt.xlabel(
    "Period"
)

plt.ylabel(
    "Total ED Visits"
)

for bar, value in zip(
    bars,
    visit_summary
):

    plt.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height(),
        f"{value:,.0f}",
        ha="center",
        va="bottom"
    )

plt.tight_layout()

plt.show()

In [ ]:
# WEIGHTED MEDIAN LOS BY PERIOD

weighted_los = period_summary[
    "weighted_median_los"
].values

plt.figure(figsize=(10, 6))

bars = plt.bar(
    period_labels,
    weighted_los
)

plt.title(
    "Weighted Median Reported ED Length of Stay by Period"
)

plt.xlabel(
    "Period"
)

plt.ylabel(
    "Weighted Median LOS (Minutes)"
)

for bar, value in zip(
    bars,
    weighted_los
):

    plt.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height(),
        f"{value:.0f}",
        ha="center",
        va="bottom"
    )

plt.tight_layout()

plt.show()

In [ ]:
# WEIGHTED MEAN VS WEIGHTED MEDIAN LOS

mean_los = period_summary[
    "weighted_mean_los"
].values

median_los = period_summary[
    "weighted_median_los"
].values

x = np.arange(
    len(period_labels)
)

width = 0.36

plt.figure(figsize=(10, 6))

plt.bar(
    x - width / 2,
    mean_los,
    width,
    label="Weighted Mean"
)

plt.bar(
    x + width / 2,
    median_los,
    width,
    label="Weighted Median"
)

plt.xticks(
    x,
    period_labels
)

plt.xlabel(
    "Period"
)

plt.ylabel(
    "LOS (Minutes)"
)

plt.title(
    "Weighted Mean and Median Reported ED LOS by Period"
)

plt.legend()

plt.tight_layout()

plt.show()

In [ ]:
# LOS DISTRIBUTION BY PERIOD

box_data = [
    pre_pandemic_df["median_los_minutes"].dropna().values,
    pandemic_df["median_los_minutes"].dropna().values
]

plt.figure(figsize=(10, 6))

plt.boxplot(
    box_data,
    labels=period_labels,
    showmeans=True
)

plt.title(
    "Distribution of Reported Median ED LOS by Period"
)

plt.xlabel(
    "Period"
)

plt.ylabel(
    "Reported Median LOS (Minutes)"
)

plt.tight_layout()

plt.show()

In [ ]:
# REPORTED MEDIAN LOS TREND BY FISCAL YEAR

year_trend = (
    h2_df
    .groupby(
        ["fiscal_year_start", "fiscal_year"]
    )
    .apply(
        lambda x: pd.Series({

            "weighted_mean_los":
                weighted_mean(
                    x["median_los_minutes"],
                    x["ed_visits"]
                ),

            "weighted_median_los":
                weighted_median(
                    x["median_los_minutes"],
                    x["ed_visits"]
                ),

            "total_ed_visits":
                x["ed_visits"].sum()

        }),
        include_groups=False
    )
    .reset_index()
    .sort_values("fiscal_year_start")
)

plt.figure(figsize=(13, 7))

plt.plot(
    year_trend["fiscal_year_start"],
    year_trend["weighted_mean_los"],
    marker="o",
    label="Weighted Mean LOS"
)

plt.plot(
    year_trend["fiscal_year_start"],
    year_trend["weighted_median_los"],
    marker="o",
    label="Weighted Median LOS"
)

plt.axvline(
    PANDEMIC_START,
    linestyle="--",
    label="Pandemic Fiscal Year Start"
)

plt.title(
    "Reported ED LOS Trend by Fiscal Year"
)

plt.xlabel(
    "Fiscal Year Start"
)

plt.ylabel(
    "Reported LOS (Minutes)"
)

plt.legend()

plt.grid(
    alpha=0.3
)

plt.tight_layout()

plt.show()

In [ ]:
# ESTIMATED RESOURCE BURDEN INDEX BY PERIOD

h2_erbi = pd.Series({

    "Pre-Pandemic":
        (
            pre_pandemic_df["ed_visits"] *
            pre_pandemic_df["median_los_minutes"]
        ).sum(),

    "Pandemic 2020-2021":
        (
            pandemic_df["ed_visits"] *
            pandemic_df["median_los_minutes"]
        ).sum()

})

plt.figure(figsize=(10, 6))

bars = plt.bar(
    h2_erbi.index,
    h2_erbi.values
)

plt.title(
    "Estimated Resource Burden Index by Period"
)

plt.xlabel(
    "Period"
)

plt.ylabel(
    "Estimated Resource Burden Index"
)

for bar, value in zip(
    bars,
    h2_erbi.values
):

    plt.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height(),
        f"{value / 1e9:.2f}B",
        ha="center",
        va="bottom",
        fontsize=9
    )

plt.tight_layout()

plt.show()

In [ ]:
# ED VISIT SHARE BY PERIOD

visit_share = (
    visit_summary /
    visit_summary.sum()
) * 100

plt.figure(figsize=(9, 7))

plt.pie(
    visit_share,
    labels=period_labels,
    autopct="%1.1f%%",
    startangle=90
)

plt.title(
    "Share of Reported ED Visits by Period"
)

plt.tight_layout()

plt.show()

In [ ]:
# ED VISIT VOLUME VS LOS BY FISCAL YEAR

scatter_data = year_trend.copy()

plt.figure(figsize=(10, 7))

plt.scatter(
    scatter_data["total_ed_visits"],
    scatter_data["weighted_median_los"],
    s=120
)

for _, row in scatter_data.iterrows():

    plt.annotate(
        str(int(row["fiscal_year_start"])),
        (
            row["total_ed_visits"],
            row["weighted_median_los"]
        ),
        xytext=(8, 8),
        textcoords="offset points"
    )

plt.xlabel(
    "Total ED Visits"
)

plt.ylabel(
    "Weighted Median LOS (Minutes)"
)

plt.title(
    "ED Visit Volume vs Reported Median LOS by Fiscal Year"
)

plt.grid(
    alpha=0.3
)

plt.tight_layout()

plt.show()

In [ ]:
# H2 STATISTICAL RESULT

decision = (
    "Reject H₀"
    if p_value < ALPHA
    else
    "Fail to Reject H₀"
)

plt.figure(figsize=(10, 5))

plt.axis("off")

plt.text(
    0.05,
    0.78,
    "H2 — Pandemic Period vs Reported Median ED LOS",
    fontsize=16,
    fontweight="bold"
)

plt.text(
    0.05,
    0.60,
    f"Mann–Whitney U = {u_stat:.4f}",
    fontsize=13
)

plt.text(
    0.05,
    0.48,
    f"Pandemic sample size = {n1:,}",
    fontsize=13
)

plt.text(
    0.05,
    0.36,
    f"Pre-pandemic sample size = {n2:,}",
    fontsize=13
)

plt.text(
    0.05,
    0.24,
    f"P-value = {p_value:.6g}",
    fontsize=13
)

plt.text(
    0.05,
    0.12,
    f"Rank-biserial correlation = {rank_biserial:.6f}    |    Decision = {decision}",
    fontsize=13,
    fontweight="bold"
)

plt.tight_layout()

plt.show()

## Summary

This notebook used the `ED_Visits.csv` explorer dataset to compare reported median emergency department length of stay between the pandemic-affected fiscal year **2020–2021** and the preceding fiscal years.

- The workflow mirrors the H1 testing notebook: dataset inspection, validation, descriptive statistics, group summaries, formal hypothesis testing, effect size, and multiple visualizations.
- The formal test is a **two-sided Mann-Whitney U test**, because the H2 question asks whether the distributions differ without specifying an increase or decrease.
- The analysis uses only fiscal years preceding **2020–2021** for the pre-pandemic comparison.
- The final conclusion is based on whether the p-value is below the selected significance threshold of **0.05**.